In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
rich   = ["United States", "United Kingdom", "Germany", "Japan"]
rising = ["China", "India", "Sub-Saharan Africa"]
all_countries = rich + rising
 
rich_colors = {
    "United States":  "#1f77b4",
    "United Kingdom": "#b2ffff",
    "Germany":        "#2ca02c",
    "Japan":          "#98df8a",
}
rising_colors = {
    "China":              "#d62728",
    "India":              "#ff7f0e",
    "Sub-Saharan Africa": "#9467bd",
}
palette = {**rich_colors, **rising_colors}
 
annotation_years = [1980, 2000, 2020]

base_url = (
    "https://raw.githubusercontent.com/light-and-salt/"
    "World-Bank-Data-by-Indicators/master/economy-and-growth/"
)

df_raw = pd.read_csv(base_url + "economy-and-growth.csv")
df_raw.columns = df_raw.columns.str.strip()
 
year_col    = "Year"
cname_col   = "Country Name"
gdp_pc_col  = "average_value_GDP per capita (constant 2010 US$)"
gdp_cur_col = "average_value_GDP (current US$)"
growth_col  = "average_value_GDP growth (annual %)"
gdp_pc_gr   = "average_value_GDP per capita growth (annual %)"

keep = [year_col, cname_col, gdp_pc_col, gdp_cur_col, growth_col, gdp_pc_gr]
df   = df_raw[keep].copy()
 
for col in keep[2:]:
    df[col]  = pd.to_numeric(df[col], errors = "coerce")
df[year_col] = pd.to_numeric(df[year_col], errors = "coerce")
 
df = df.rename(columns = {
    gdp_pc_col: "GDP_pc",
    gdp_cur_col: "Total_GDP",
    growth_col: "GDP_growth",
    gdp_pc_gr: "GDP_pc_growth",
})
 
df_focus = df[df[cname_col].isin(all_countries)].copy()

In [3]:
df_v1 = (
    df_focus
    .groupby([cname_col, year_col], as_index = False)["GDP_pc"]
    .mean()
)

fig1 = go.Figure()

# Rich-country lines (solid)
for country in rich:
    d = (df_v1[df_v1[cname_col] == country]
        .dropna(subset = ["GDP_pc"])
        .sort_values(year_col)
    )
    
    fig1.add_trace(go.Scatter(
        x = d[year_col], y = d["GDP_pc"],
        mode = "lines", name = country,
        line = dict(color = palette[country], width = 2.5),
        legendgroup = "rich",
        legendgrouptitle_text = "High-Income",
    ))

# Rising-country lines (dashed)
for country in rising:
    d = (df_v1[df_v1[cname_col] == country]
        .dropna(subset = ["GDP_pc"])
        .sort_values(year_col)
    )
    
    fig1.add_trace(go.Scatter(
        x = d[year_col], y = d["GDP_pc"],
        mode = "lines", name = country,
        line = dict(color = rising_colors[country], width = 2.5, dash = "dash"),
        legendgroup = "rising",
        legendgrouptitle_text = "Developing",
    ))

# Shade the gap between US and China
us  = df_v1[df_v1[cname_col] == "United States"].dropna(subset = ["GDP_pc"]).sort_values(year_col)
chn = df_v1[df_v1[cname_col] == "China"].dropna(subset = ["GDP_pc"]).sort_values(year_col)
gap_df = us[[year_col, "GDP_pc"]].merge(
    chn[[year_col, "GDP_pc"]], on = year_col, suffixes = ("_us", "_cn")
)

fig1.add_trace(go.Scatter(
    x = pd.concat([gap_df[year_col], gap_df[year_col][::-1]]),
    y = pd.concat([gap_df["GDP_pc_us"], gap_df["GDP_pc_cn"][::-1]]),
    fill = "toself",
    fillcolor = "rgba(31,119,180,0.08)",
    line = dict(color = "rgba(0,0,0,0)"),
    name = "US–China gap",
    hoverinfo = "skip",
))

# Dollar-gap annotations at key years
for yr in annotation_years:
    row = gap_df[gap_df[year_col] == yr]
    if row.empty:
        continue
    gap = row["GDP_pc_us"].values[0] - row["GDP_pc_cn"].values[0]
    mid = (row["GDP_pc_us"].values[0] + row["GDP_pc_cn"].values[0]) / 2
    fig1.add_annotation(
        x = yr, y = mid,
        text = f"Gap: ${gap:,.0f}",
        showarrow = True, arrowhead=2, arrowcolor = "#aaa",
        font = dict(size = 11, color = "#333"),
        bgcolor = "rgba(255,255,255,0.85)",
        bordercolor = "#aaa", borderwidth = 1,
    )

fig1.update_layout(
    title = dict(
        text = (
            "<b>The Wealth Gap</b> — GDP Per Capita, 1960–2020 "
            "(constant 2010 USD)<br>"
            "<sub>Even as growth <i>rates</i> converge, the absolute "
            "dollar gap has <i>widened</i></sub>"
        ),
        font = dict(size = 16),
    ),
    xaxis = dict(title="Year", tickmode = "linear", dtick = 10),
    yaxis = dict(
        title = "GDP per Capita (constant 2010 USD)",
        tickprefix = "$", tickformat = ",.0f",
        rangemode = "tozero",
    ),
    hovermode = "x unified",
    legend = dict(groupclick = "toggleitem"),
    template = "plotly_white",
    height = 520,
    width = 1300,
)

fig1.show()